# 🇻🇳 Vietnamese Fake News: Crawling & Preprocessing Pipeline

**Complete workflow for Vietnamese news data collection and processing**

## 📋 Pipeline Overview

1. **Web Crawling** - Extract articles from VifactCheck dataset
2. **Data Preprocessing** - Process text (PhoBERT) and images (ResNet)
3. **Dataset Creation** - Generate ML-ready datasets
4. **Analysis & Export** - Visualize and save results


In [1]:
# Setup
import sys
import os
import json
import asyncio
import nest_asyncio
import pandas as pd
import numpy as np
import torch
import warnings
from pathlib import Path
from typing import List, Dict, Tuple
from datetime import datetime

warnings.filterwarnings("ignore")
nest_asyncio.apply()

# Set OPENSSL_CONF environment variable (required for crawler)
os.environ["OPENSSL_CONF"] = "openssl.cnf"

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

print(f"✅ Ready: {project_root}")

✅ Ready: /Volumes/Private/Thesis_2026/fake-new-detection


## 🕷️ Part 1: Web Crawling (Using CrawlerFactory)

**Based on `src/test_crawler.py` - crawls from Hugging Face dataset**


In [2]:
# Import crawler modules
try:
    from src.crawler.crawler_factory import CrawlerFactory
    from src.processing.dataset_handler import DatasetHandler
    from src.helpers.logger import logger

    print("✅ Crawler modules imported successfully")
except ImportError as e:
    print(f"❌ Error importing crawler modules: {e}")
    print("Make sure you're in the project root directory")

❌ Error importing crawler modules: No module named 'exceptions'
Make sure you're in the project root directory


In [ ]:
{
    "cell_type": "code",
    "execution_count": null,
    "metadata": {},
    "outputs": [],
    "source": [
        "# Crawling configuration (matching test_crawler.py)\n",
        "CRAWLING_CONFIG = {\n",
        '    "dataset_name": "tranthaihoa/vifactcheck",\n',
        '    "url_column": "Url",\n',
        '    "splits": ["train", "test", "dev"],\n',
        '    "url_limit": None,  # Set to None to crawl ALL URLs (no limit)\n',
        '    "cache_dir": "./data/caches",\n',
        '    "output_dir": "./src/data/json"\n',
        "}\n",
        "\n",
        "# Create necessary directories\n",
        'os.makedirs(CRAWLING_CONFIG["cache_dir"], exist_ok=True)\n',
        'os.makedirs(CRAWLING_CONFIG["output_dir"], exist_ok=True)\n',
        "\n",
        'print("🔧 Crawling configured:")\n',
        "for key, value in CRAWLING_CONFIG.items():\n",
        '    print(f"  • {key}: {value}")\n',
        "\n",
        "print(f\"\\n🚀 Ready to crawl ALL URLs from {CRAWLING_CONFIG['dataset_name']}\")\n",
        'print(f"📊 This will process train/dev/test splits without limits")',
    ],
}

🔧 Crawling configured:
  • dataset_name: tranthaihoa/vifactcheck
  • url_column: Url
  • splits: ['train', 'test', 'dev']
  • url_limit: 5
  • cache_dir: ./data/caches
  • output_dir: ./src/data/json


In [4]:
async def crawl_vifactcheck_dataset():
    """Crawl URLs from VifactCheck dataset using CrawlerFactory"""
    try:
        dataset_name = CRAWLING_CONFIG["dataset_name"]
        url_column = CRAWLING_CONFIG["url_column"]
        splits = CRAWLING_CONFIG["splits"]
        url_limit = CRAWLING_CONFIG["url_limit"]

        crawling_results = []

        for split in splits:
            print(f"\n--- Processing split: {split} ---")

            # Setup crawler factory
            crawler_factory = CrawlerFactory(
                cache_filename=f"{CRAWLING_CONFIG['cache_dir']}/crawling_status_{split}.json",
                failed_log_filename=f"{CRAWLING_CONFIG['cache_dir']}/failed_urls_{split}.json",
            )

            # Clear cache for fresh crawl (optional)
            if crawler_factory.check_cache_file_exists():
                print(f"Clearing cache for split '{split}'...")
                crawler_factory.clear_cache()

            # Load URLs from dataset
            dataset_handler = DatasetHandler(dataset_name)
            urls_to_crawl = dataset_handler.get_urls_from_split(
                split=split, url_column=url_column, limit=url_limit
            )

            if urls_to_crawl:
                print(f"Found {len(urls_to_crawl)} URLs to crawl for {split}")

                # Setup output filename
                output_filename = (
                    f"news_data_{dataset_name.split('/')[-1]}_{split}.json"
                )
                output_path = os.path.join(
                    CRAWLING_CONFIG["output_dir"], output_filename
                )

                # Start crawling
                print(f"Starting crawl for {split} split...")
                await crawler_factory.crawl_and_save_all(
                    urls_to_crawl, output_filename, format_name="custom"
                )

                result = {
                    "split": split,
                    "urls_found": len(urls_to_crawl),
                    "output_file": output_path,
                    "timestamp": datetime.now().isoformat(),
                }
                crawling_results.append(result)

                print(f"✅ Completed {split} split")
            else:
                print(f"❌ No URLs found for split {split}")

        return crawling_results

    except Exception as e:
        print(f"❌ Crawling error: {e}")
        import traceback

        traceback.print_exc()
        return []


# Test function definition
print("🔧 Crawl function defined successfully")

🔧 Crawl function defined successfully


In [ ]:
{
    "cell_type": "code",
    "execution_count": null,
    "metadata": {},
    "outputs": [],
    "source": [
        "# Execute crawling - This will crawl ALL URLs from the VifactCheck dataset\n",
        "# WARNING: This may take a long time depending on dataset size\n",
        "# Uncomment the lines below to start crawling:\n",
        "\n",
        "# crawling_results = await crawl_vifactcheck_dataset()\n",
        "# \n",
        "# if crawling_results:\n",
        '#     print("\\n✅ Full crawling completed!")\n',
        "#     total_urls = sum(result['urls_found'] for result in crawling_results)\n",
        '#     print(f"📊 Total URLs processed: {total_urls}")\n',
        "#     for result in crawling_results:\n",
        "#         print(f\"📁 {result['split']}: {result['urls_found']} URLs → {result['output_file']}\")\n",
        "# else:\n",
        '#     print("❌ No crawling results")\n',
        "\n",
        'print("🚀 Ready to crawl ALL URLs from VifactCheck dataset")\n',
        'print("💡 Uncomment the crawling lines above to start the full crawl")',
    ],
}

## 🔄 Part 2: Data Preprocessing


In [6]:
# Preprocessing config
PREPROCESSING_CONFIG = {
    "text_model": "vinai/phobert-base",
    "image_model": "resnet18",
    "language": "vi",
    "max_length": 512,
    "batch_size": 16,
    "save_format": "pkl",
}

# Device detection
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Using device: {device}")

🔧 Using device: mps


In [7]:
# Initialize preprocessor
try:
    from src.preprocessing import CombinedPreprocessor

    preprocessor = CombinedPreprocessor(
        text_model_name=PREPROCESSING_CONFIG["text_model"],
        image_model_name=PREPROCESSING_CONFIG["image_model"],
        language=PREPROCESSING_CONFIG["language"],
        max_text_length=PREPROCESSING_CONFIG["max_length"],
        device=device,
    )
    print("✅ Preprocessor ready")
except Exception as e:
    print(f"❌ Preprocessor error: {e}")

✅ Preprocessor ready


In [8]:
def load_crawled_data():
    """Load crawled data from the JSON output files"""
    # Look for crawled data in the expected output directory
    data_dir = Path(CRAWLING_CONFIG["output_dir"])

    if not data_dir.exists():
        print(f"❌ Data directory not found: {data_dir}")
        return []

    # Find all JSON files matching the pattern
    json_files = list(data_dir.glob("news_data_vifactcheck_*.json"))

    all_articles = []

    for json_file in json_files:
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                articles = json.load(f)

            # Add source information
            for article in articles:
                article["source_file"] = str(json_file)
                article["source_split"] = json_file.stem.split("_")[
                    -1
                ]  # Extract split from filename

            all_articles.extend(articles)
            print(f"✅ Loaded {len(articles)} articles from {json_file.name}")

        except Exception as e:
            print(f"❌ Error loading {json_file}: {e}")

    print(f"\n📊 Total articles loaded: {len(all_articles)}")
    return all_articles


# Load crawled data
crawled_articles = load_crawled_data()

if crawled_articles:
    # Display sample article structure
    print("\n📄 Sample article structure:")
    sample = crawled_articles[0]
    for key, value in sample.items():
        if isinstance(value, str) and len(value) > 100:
            print(f"  • {key}: {value[:100]}...")
        else:
            print(f"  • {key}: {value}")


📊 Total articles loaded: 0


In [9]:
def prepare_data_for_preprocessing(
    articles: List[Dict],
) -> Tuple[List[str], List[str], List[int]]:
    """
    Prepare crawled VifactCheck data for preprocessing

    Returns:
        Tuple of (texts, image_paths, labels)
    """
    texts = []
    image_paths = []
    labels = []

    for i, article in enumerate(articles):
        # Extract text content (try multiple fields)
        text = (
            article.get("text", "")
            or article.get("content", "")
            or article.get("title", "")
            or article.get("description", "")
        )

        if text.strip():
            texts.append(text.strip())

            # Extract image path
            images = article.get("images", [])
            if images and len(images) > 0:
                # Handle different image formats
                if isinstance(images[0], str):
                    image_path = images[0]
                elif isinstance(images[0], dict):
                    image_path = (
                        images[0].get("url", "")
                        or images[0].get("path", "")
                        or images[0].get("src", "")
                    )
                else:
                    image_path = str(images[0])

                image_paths.append(image_path if image_path else None)
            else:
                image_paths.append(None)

            # Extract label (VifactCheck uses 'label' field)
            label = article.get("label", 0)
            if isinstance(label, str):
                # Convert string labels to integers
                label = 1 if label.lower() in ["fake", "false", "1"] else 0
            labels.append(int(label))

    return texts, image_paths, labels


# Prepare data
if crawled_articles:
    texts, image_paths, labels = prepare_data_for_preprocessing(crawled_articles)

    print(f"📊 Data preparation completed:")
    print(f"  • Text samples: {len(texts)}")
    print(f"  • Image paths: {len(image_paths)}")
    print(f"  • Labels: {len(labels)}")
    print(f"  • Valid images: {sum(1 for path in image_paths if path is not None)}")

    if labels:
        label_counts = {}
        for label in labels:
            label_counts[label] = label_counts.get(label, 0) + 1
        print(f"  • Label distribution: {label_counts}")
else:
    print("❌ No crawled data available for preparation")

❌ No crawled data available for preparation


In [10]:
def create_placeholders(image_paths):
    """Create placeholder images for missing ones"""
    from PIL import Image

    placeholder_dir = Path("./placeholder_images")
    placeholder_dir.mkdir(exist_ok=True)

    processed_paths = []
    for i, img_path in enumerate(image_paths):
        if not img_path or not os.path.exists(img_path):
            placeholder_path = placeholder_dir / f"placeholder_{i}.jpg"
            if not placeholder_path.exists():
                placeholder_array = np.random.randint(
                    128, 200, (224, 224, 3), dtype=np.uint8
                )
                placeholder_image = Image.fromarray(placeholder_array)
                placeholder_image.save(placeholder_path)
            processed_paths.append(str(placeholder_path))
        else:
            processed_paths.append(img_path)

    return processed_paths


# Create placeholders if we have data
if "texts" in locals() and texts:
    processed_image_paths = create_placeholders(image_paths)
    print(f"✅ Created {len(processed_image_paths)} image paths")

In [11]:
# Run preprocessing
if "preprocessor" in locals() and "texts" in locals() and texts:
    try:
        print("🔄 Starting preprocessing...")

        # Create output directory
        processed_dir = "./processed_data/crawled"
        os.makedirs(processed_dir, exist_ok=True)

        # Process dataset
        dataset, metadata = preprocessor.preprocess_dataset(
            texts=texts,
            image_paths=processed_image_paths,
            labels=labels,
            save_dir=processed_dir,
            save_format=PREPROCESSING_CONFIG["save_format"],
            batch_size=PREPROCESSING_CONFIG["batch_size"],
        )

        print("✅ Preprocessing complete!")
        print(f"📊 Dataset: {len(dataset)} samples")
        print(f"📝 Text shape: {metadata['text_feature_shape']}")
        print(f"🖼️ Image shape: {metadata['image_feature_shape']}")

    except Exception as e:
        print(f"❌ Preprocessing error: {e}")
else:
    print("❌ Cannot preprocess - missing data or preprocessor")

❌ Cannot preprocess - missing data or preprocessor


## 📊 Part 3: Analysis & Visualization


In [12]:
# Quick analysis
if "texts" in locals() and texts:
    import matplotlib.pyplot as plt
    from collections import Counter

    # Text statistics
    text_lengths = [len(text.split()) for text in texts]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Text length distribution
    ax1.hist(text_lengths, bins=20, alpha=0.7)
    ax1.set_title("Text Length Distribution")
    ax1.set_xlabel("Words")
    ax1.set_ylabel("Frequency")

    # Label distribution
    label_counts = Counter(labels)
    ax2.bar(["Real", "Fake"], [label_counts.get(0, 0), label_counts.get(1, 0)])
    ax2.set_title("Label Distribution")
    ax2.set_ylabel("Count")

    plt.tight_layout()
    plt.show()

    print(f"📊 Total samples: {len(texts)}")
    print(f"📝 Avg text length: {np.mean(text_lengths):.1f} words")
    print(f"🏷️ Label distribution: {dict(label_counts)}")
else:
    print("❌ No data to analyze")

❌ No data to analyze


## 💾 Part 4: Export Results


In [13]:
def export_results():
    """Export processed data in multiple formats"""
    try:
        processed_dir = "./processed_data/crawled"
        combined_path = os.path.join(processed_dir, "combined_dataset.pkl")

        if os.path.exists(combined_path):
            # Load data
            text_features, image_features, labels = preprocessor.load_combined_dataset(
                combined_path
            )

            # Export as NPZ
            npz_path = os.path.join(processed_dir, "dataset.npz")
            np.savez(
                npz_path,
                text_features=text_features,
                image_features=image_features,
                labels=labels,
            )

            # Export metadata CSV
            metadata_df = pd.DataFrame(
                {
                    "sample_id": range(len(labels)),
                    "label": labels,
                    "text_shape": [
                        text_features[i].shape for i in range(len(text_features))
                    ],
                }
            )

            csv_path = os.path.join(processed_dir, "metadata.csv")
            metadata_df.to_csv(csv_path, index=False)

            print(f"✅ Exported: NPZ ({npz_path}), CSV ({csv_path})")
        else:
            print("❌ No processed data found")

    except Exception as e:
        print(f"❌ Export error: {e}")


# Export results
export_results()

❌ No processed data found


## 🎯 Summary

### ✅ What's Ready:

- **Web Crawling**: VifactCheck dataset crawler using CrawlerFactory
- **Preprocessing**: PhoBERT + ResNet pipeline
- **Analysis**: Text and label visualization
- **Export**: Multiple format support

### 🚀 How to Use:

1. **Crawl**: Uncomment `crawling_results = await crawl_vifactcheck_dataset()`
2. **Process**: Run preprocessing cells sequentially
3. **Analyze**: View visualizations
4. **Export**: Save for training

### 📁 Output Structure:

```
./src/data/json/          # Crawled articles
./data/caches/           # Crawler cache
./processed_data/crawled/ # Processed ML data
./placeholder_images/    # Generated placeholders
```


In [14]:
# Pipeline status check
def check_status():
    print("🚀 VifactCheck Pipeline Status:")
    print("=" * 40)

    # Check crawled data
    data_dir = Path(CRAWLING_CONFIG["output_dir"])
    if data_dir.exists():
        json_files = list(data_dir.glob("news_data_vifactcheck_*.json"))
        print(f"📰 Crawled data: {len(json_files)} files found")
        for f in json_files:
            print(f"  • {f.name}")
    else:
        print("📰 Crawled data: Not found")

    # Check cache files
    cache_dir = Path(CRAWLING_CONFIG["cache_dir"])
    if cache_dir.exists():
        cache_files = list(cache_dir.glob("*.json"))
        print(f"💾 Cache files: {len(cache_files)} files")

    # Check processed data
    processed_dir = Path("./processed_data/crawled")
    if processed_dir.exists():
        files = list(processed_dir.glob("*"))
        print(f"🔄 Processed data: {len(files)} files")
    else:
        print("🔄 Processed data: Not found")

    # Check preprocessor
    if "preprocessor" in locals():
        print("✅ Preprocessor: Initialized")
    else:
        print("❌ Preprocessor: Not initialized")

    print("\n🎯 Next steps:")
    print("1. Run crawling (uncomment crawling line)")
    print("2. Load and prepare data")
    print("3. Execute preprocessing with PhoBERT + ResNet")
    print("4. Analyze results and export for training")

    print("\n📁 Expected structure:")
    print("  ./src/data/json/news_data_vifactcheck_*.json")
    print("  ./data/caches/crawling_status_*.json")
    print("  ./processed_data/crawled/")


check_status()

🚀 VifactCheck Pipeline Status:
📰 Crawled data: 0 files found
💾 Cache files: 0 files
🔄 Processed data: Not found
❌ Preprocessor: Not initialized

🎯 Next steps:
1. Run crawling (uncomment crawling line)
2. Load and prepare data
3. Execute preprocessing with PhoBERT + ResNet
4. Analyze results and export for training

📁 Expected structure:
  ./src/data/json/news_data_vifactcheck_*.json
  ./data/caches/crawling_status_*.json
  ./processed_data/crawled/
